In [3]:
!pip install langdetect bertopic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=b6807cb9fcc2860cbd196f087972b28fce862cad671e1a4feead2303100b8e11
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [1]:
#!/bin/bash
!curl -L -o course-reviews-on-coursera.zip\
  https://www.kaggle.com/api/v1/datasets/download/imuhammad/course-reviews-on-coursera

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 33.1M  100 33.1M    0     0  36.9M      0 --:--:-- --:--:-- --:--:--  104M


In [2]:
!unzip course-reviews-on-coursera.zip

Archive:  course-reviews-on-coursera.zip
  inflating: Coursera_courses.csv    
  inflating: Coursera_reviews.csv    


In [4]:
import pandas as pd

df = pd.read_csv('Coursera_reviews.csv')

In [6]:
from langdetect import detect

text_to_detect = "Good course very helpfull"

eng = detect(text_to_detect)

In [9]:
df.loc[0]

,0
reviews,"Pretty dry, but I was able to pass with just t..."
reviewers,By Robert S
date_reviews,"Feb 12, 2020"
rating,4
course_id,google-cbrs-cpi-training


In [7]:
detect(df.loc[0,'reviews']) == eng

True

In [11]:
df['reviews'] = df['reviews'].astype(str)

In [13]:
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

# Download necessary NLTK resource packages (only required once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [19]:
def tokenizz(val):
  val = val.lower()

  return val.split()

In [20]:
tokenizz(df.loc[0,'reviews'])

['pretty',
 'dry,',
 'but',
 'i',
 'was',
 'able',
 'to',
 'pass',
 'with',
 'just',
 'two',
 'complete',
 'watches',
 'so',
 "i'm",
 'happy',
 'about',
 'that.',
 'as',
 'usual',
 'there',
 'were',
 'some',
 'questions',
 'on',
 'the',
 'final',
 'exam',
 'that',
 'were',
 'no',
 'where',
 'in',
 'the',
 'course,',
 'which',
 'is',
 'annoying',
 'but',
 'far',
 'better',
 'than',
 'many',
 'microsoft',
 'tests',
 'i',
 'have',
 'taken.',
 'never',
 'found',
 'the',
 'suplimental',
 'material',
 'that',
 'the',
 'course',
 'references...',
 'but',
 'who',
 'cares...',
 'i',
 'passed!']

In [21]:
df['tokens'] = df['reviews'].apply(tokenizz)

In [22]:
df['flag'] = df['tokens'].apply(lambda x: len(x) < 3)

In [23]:
df = df.drop(df[df['flag']].index)

In [24]:
df['flag'].value_counts()

,count
flag,
False,1275037


In [26]:
def detect_eng(text, fallback=False):
    try:
        return detect(text) != eng
    except Exception:
        return fallback

In [27]:
df['flag'] = df['reviews'].apply(lambda x: detect_eng(x, True))

In [28]:
df = df.drop(df[df['flag']].index)

In [29]:
from google.colab import files

uploaded = files.upload()

Saving cleaned_reviews.zip to cleaned_reviews.zip


In [30]:
!unzip cleaned_reviews.zip

Archive:  cleaned_reviews.zip
  inflating: cleaned_reviews.csv     


In [32]:
df = df.join(pd.read_csv('cleaned_reviews.csv'), how='left')

In [33]:
df.sample(1)

,reviews,reviewers,date_reviews,rating,course_id,tokens,flag,Unnamed: 0,cleaned_text
272682,Amazing course to get to the point insights on...,By Siddharth J,"Oct 14, 2018",5,gcp-fundamentals,"[amazing, course, to, get, to, the, point, ins...",False,272682,"['amaze', 'course', 'get', 'point', 'insight',..."


In [35]:
df = df.groupby('rating', group_keys=False).apply(
    lambda x: x.sample(n=14000, random_state=42)
)

/tmp/ipykernel_888/76486300.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('rating', group_keys=False).apply(


In [36]:
df.rating.value_counts()

,count
rating,
1,14000
2,14000
3,14000
4,14000
5,14000


In [37]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

In [40]:
from bertopic import BERTopic
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN
from sentence_transformers import SentenceTransformer

# 1. Embeddings (Automatically uses CUDA if PyTorch detects a GPU)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device='cuda')

# 2. CUDA-Accelerated Dimensionality Reduction
umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, random_state=42)

# 3. CUDA-Accelerated Clustering
hdbscan_model = HDBSCAN(min_samples=10, gen_min_span_tree=True, prediction_data=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [57]:
topic_list = []

In [58]:
# Run the pipeline
for i in range(1,6):
  topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    # calculate_probabilities=False # Set False on large datasets to save GPU memory
  )

  topics, probs = topic_model.fit_transform(df.loc[df['rating'] == i,'cleaned_text'].tolist())

  topic_list.append(topic_model.get_topic_info())

In [69]:
topic_list[1][:10]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1424,-1_marketing_lecture_experience_also,"[marketing, lecture, experience, also, answer,...","[['course', ""n't"", 'good', 'first', 'one', 'hr..."
1,0,83,0_french_student_receive_article,"[french, student, receive, article, forum, rai...","[['work', 'scientific', 'research', 'field', '..."
2,1,75,1_excel_analysis_spreadsheet_formula,"[excel, analysis, spreadsheet, formula, regres...","[['learn', 'helpful', 'tip', 'analyze', 'data'..."
3,2,72,2_ml_ai_deeplearning_algorithm,"[ml, ai, deeplearning, algorithm, intent, cont...","[['true', 'kill', 'intension', 'explore', 'fie..."
4,3,71,3_exercise_soak_sps_manger,"[exercise, soak, sps, manger, inbetween, facil...","[['simulation', 'specialist', 'center', 'mange..."
5,4,66,4_scholar_alongside_editor_cyber,"[scholar, alongside, editor, cyber, hope, shor...","[['content', 'quite', 'interest', 'felt', 'val..."
6,5,58,5_peer_review_reviewer_submission,"[peer, review, reviewer, submission, dislike, ...","[['good', 'course', 'peer', 'review', 'process..."
7,6,53,6_manager_improve_cohesive_design,"[manager, improve, cohesive, design, project, ...","[['project', 'manager', 'hop', 'spend', 'valua..."
8,7,52,7_ibm_clearn_disappointined_alternatively,"[ibm, clearn, disappointined, alternatively, a...","[['ibm', 'watson', 'section', 'totally', 'date..."
9,8,52,8_python_geoson_waist_skicit,"[python, geoson, waist, skicit, reinstall, try...","[['lab', 'always', 'function', 'properly', 'fi..."
